# Encoder & Decoder - 从简化到完整的 Transformer 核心组件

## 目标：彻底搞懂 Transformer 的 Encoder 和 Decoder

1. **Transformer 整体架构** —— Encoder-Decoder 框架概览
2. **简化版 Self-Attention** —— 纯 numpy 实现，无参数
3. **完整 Self-Attention** —— 加入可学习权重矩阵
4. **Multi-Head Attention** —— 多头并行计算
5. **Encoder Block** —— 残差连接 + LayerNorm + FFN
6. **Decoder 核心** —— Masked Self-Attention + Cross-Attention
7. **完整 Transformer** —— Encoder + Decoder 端到端演示

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("所有依赖已加载！")
print(f"numpy version: {np.__version__}")
print(f"torch version: {torch.__version__}")

---

# PART 1: Transformer 整体架构概览

## Encoder-Decoder 框架

Transformer 的核心是一个**编码器-解码器（Encoder-Decoder）**架构：

```
                    ┌──────────────┐
                    │   Encoder    │
  输入文本（源语言）→│  × N 层     │
                    │              │
                    └──────┬───────┘
                           │
                           ↓
              ┌────────────┴────────────┐
              │     Cross-Attention     │
              │   Decoder "看" Encoder   │
              └────────────┬────────────┘
                           │
                    ┌──────┴───────┐
                    │   Decoder    │
  输入文本（目标语言）→│  × N 层     │→ 输出文本
                    │              │
                    └──────────────┘
```

## 核心组件

### Encoder 组件

每个 Encoder Block 包含：
1. **Multi-Head Self-Attention** —— 让每个 token 关注序列中所有其他 token
2. **残差连接 + LayerNorm** —— 稳定训练
3. **Feed-Forward Network (FFN)** —— 对每个 token 独立进行非线性变换

### Decoder 组件

每个 Decoder Block 包含：
1. **Masked Multi-Head Self-Attention** —— 只能关注前面的 token（因果约束）
2. **残差连接 + LayerNorm**
3. **Multi-Head Cross-Attention** —— Decoder 关注 Encoder 的输出
4. **残差连接 + LayerNorm**
5. **Feed-Forward Network**

---

# PART 2: 简化版 Self-Attention（纯 numpy，无参数）

## Self-Attention 的核心思想

Self-Attention 的核心是让每个 token 计算与其他所有 token 的**相似度**，然后基于相似度对其他 token 的信息进行**加权求和**。

## 公式

Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

## 简化版本

为了理解原理，我们先实现一个最简单的版本：Q=K=V=X，没有可学习的权重矩阵。

In [ ]:
# 简化版 Self-Attention（Q=K=V=X）

def simple_self_attention(X):
    """
    极简化 Self-Attention（无参数版本）
    
    Args:
        X: 输入矩阵，shape = (seq_len, d_model)
    
    Returns:
        输出矩阵，shape = (seq_len, d_model)
    """
    seq_len, d_model = X.shape
    
    # Step 1: 计算注意力分数（QK^T / sqrt(d_k)）
    # Q=K=X，所以 scores = X @ X.T
    scores = X @ X.T / math.sqrt(d_model)
    print(f"Step 1: scores shape = {scores.shape}")
    print(f"        scores:\n{np.round(scores, 4)}")

    # Step 2: 对分数做 softmax，得到注意力权重
    exp_scores = np.exp(scores - scores.max(axis=-1, keepdims=True))  # 数值稳定
    attn_weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    print(f"\nStep 2: attn_weights shape = {attn_weights.shape}")
    print(f"          attn_weights:\n{np.round(attn_weights, 4)}")

    # Step 3: 用注意力权重对 V 进行加权求和
    # V=X，所以 output = attn_weights @ X
    output = attn_weights @ X
    print(f"\nStep 3: output shape = {output.shape}")

    return output, attn_weights

In [ ]:
# 测试简化版 Self-Attention

# 模拟 3 个 token 的嵌入向量（d_model=4）
np.random.seed(42)
X = np.random.randn(3, 4) * 0.1

print("=" * 60)
print("输入矩阵 X（3个token，每个4维）")
print("=" * 60)
print(f"X shape: {X.shape}")
print(f"X:\n{np.round(X, 4)}")

# 运行 Self-Attention
output, attn_weights = simple_self_attention(X)

print("\n" + "=" * 60)
print("输出结果")
print("=" * 60)
print(f"output:\n{np.round(output, 4)}")

print("\n" + "=" * 60)
print("注意力权重解读：")
print("=" * 60)
print(f"attn_weights[i, j] 表示第 i 个 token 对第 j 个 token 的关注度")
print(f"\n例如：")
print(f"  token 0 对 token 0 的关注度: {attn_weights[0, 0]:.4f}")
print(f"  token 0 对 token 1 的关注度: {attn_weights[0, 1]:.4f}")
print(f"  token 0 对 token 2 的关注度: {attn_weights[0, 2]:.4f}")
print(f"  → token 0 的输出 = {attn_weights[0, 0]:.2f}×X[0] + {attn_weights[0, 1]:.2f}×X[1] + {attn_weights[0, 2]:.2f}×X[2]")

---

# PART 3: 完整 Self-Attention（加入可学习权重）

## 真实的 Self-Attention

真实的 Self-Attention 有三个可学习的权重矩阵：

- W_Q: Query 权重矩阵，shape = (d_model, d_k)
- W_K: Key 权重矩阵，shape = (d_model, d_k)
- W_V: Value 权重矩阵，shape = (d_model, d_v)

计算过程：

1. Q = X @ W_Q
2. K = X @ W_K
3. V = X @ W_V
4. Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
5. 输出 = Attention @ W_O（可选的输出投影）

In [ ]:
# 完整 Self-Attention（带可学习权重）

class SelfAttention(nn.Module):
    """
    完整的 Self-Attention 模块

    Args:
        d_model: 输入维度
        d_k: Query/Key 维度
        d_v: Value 维度
    """

    def __init__(self, d_model: int, d_k: int, d_v: int):
        super().__init__()

        # 可学习的权重矩阵
        self.W_Q = nn.Linear(d_model, d_k)
        self.W_K = nn.Linear(d_model, d_k)
        self.W_V = nn.Linear(d_model, d_v)
        self.W_O = nn.Linear(d_v, d_model)  # 输出投影

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """
        Args:
            X: 输入张量，shape = (batch_size, seq_len, d_model)
        
        Returns:
            输出张量，shape = (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, d_model = X.shape

        # Step 1: 计算 Q, K, V
        Q = self.W_Q(X)  # (B, T, d_k)
        K = self.W_K(X)  # (B, T, d_k)
        V = self.W_V(X)  # (B, T, d_v)

        print(f"Step 1: Q shape = {Q.shape}, K shape = {K.shape}, V shape = {V.shape}")

        # Step 2: 计算注意力分数 QK^T / sqrt(d_k)
        # K^T: (B, d_k, T)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(K.shape[-1])
        print(f"Step 2: scores shape = {scores.shape}")

        # Step 3: softmax 得到注意力权重
        attn_weights = F.softmax(scores, dim=-1)
        print(f"Step 3: attn_weights shape = {attn_weights.shape}")

        # Step 4: 加权求和
        output = torch.matmul(attn_weights, V)
        print(f"Step 4: output (before W_O) shape = {output.shape}")

        # Step 5: 输出投影
        output = self.W_O(output)
        print(f"Step 5: output (after W_O) shape = {output.shape}")

        return output, attn_weights

In [ ]:
# 测试完整 Self-Attention

d_model = 16
d_k = 8
d_v = 8
batch_size = 2
seq_len = 5

# 创建 Self-Attention 模块
attn = SelfAttention(d_model, d_k, d_v)

# 模拟输入
X = torch.randn(batch_size, seq_len, d_model)

print("=" * 60)
print(f"Self-Attention 测试（d_model={d_model}, d_k={d_k}, d_v={d_v}")
print("=" * 60)
print(f"输入 X shape: {X.shape}")

# 前向传播
output, attn_weights = attn(X)

print("\n" + "=" * 60)
print("输出结果")
print("=" * 60)
print(f"输出 shape: {output.shape}")
print(f"\n注意力权重（第 1 个样本）:\n{torch.round(attn_weights[0], decimals=4)}")

---

# PART 4: Multi-Head Attention（多头注意力）

## 为什么需要多头？

单一注意力头只能学习一种类型的关系。多头注意力让模型同时学习**多种不同类型**的关系。

例如：
- 头 1：关注语法关系（主谓宾）
- 头 2：关注语义关系（同义词、反义词）
- 头 3：关注指代关系（代词指代哪个名词）

## 多头注意力的计算

1. 将输入通过 h 组不同的 W_Q/W_K/W_V 投影
2. 每组独立计算注意力
3. 将 h 个输出拼接起来
4. 通过 W_O 投影得到最终输出

In [ ]:
# Multi-Head Attention 实现

class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention 模块

    Args:
        d_model: 输入维度
        nhead: 头数
    """

    def __init__(self, d_model: int, nhead: int):
        super().__init__()

        assert d_model % nhead == 0, "d_model 必须能被 nhead 整除"

        self.nhead = nhead
        self.d_k = d_model // nhead  # 每个头的维度

        # 一次性定义所有投影矩阵
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, X: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            X: 输入张量，shape = (batch_size, seq_len, d_model)
            mask: 注意力掩码，shape = (seq_len, seq_len) 或 (batch_size, seq_len, seq_len)

        Returns:
            输出张量，shape = (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, d_model = X.shape

        # Step 1: 计算 Q, K, V
        Q = self.W_Q(X)  # (B, T, d_model)
        K = self.W_K(X)  # (B, T, d_model)
        V = self.W_V(X)  # (B, T, d_model)

        # Step 2: 重塑为多头格式
        # (B, T, d_model) → (B, T, nhead, d_k) → (B, nhead, T, d_k)
        Q = Q.view(batch_size, seq_len, self.nhead, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.nhead, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.nhead, self.d_k).transpose(1, 2)

        print(f"Step 2: Q shape after reshape = {Q.shape}")
        print(f"         (batch_size, nhead, seq_len, d_k)")

        # Step 3: 计算注意力分数
        # Q: (B, nhead, T, d_k)
        # K^T: (B, nhead, d_k, T)
        # scores: (B, nhead, T, T)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Step 4: 应用掩码（如果提供）
        if mask is not None:
            # mask: (T, T) → 扩展为 (1, 1, T, T)
            mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Step 5: softmax
        attn_weights = F.softmax(scores, dim=-1)
        print(f"Step 5: attn_weights shape = {attn_weights.shape}")

        # Step 6: 加权求和
        # (B, nhead, T, T) @ (B, nhead, T, d_k) → (B, nhead, T, d_k)
        output = torch.matmul(attn_weights, V)

        # Step 7: 重塑回原始格式
        # (B, nhead, T, d_k) → (B, T, nhead, d_k) → (B, T, d_model)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        # Step 8: 输出投影
        output = self.W_O(output)
        print(f"Step 8: final output shape = {output.shape}")

        return output, attn_weights

In [ ]:
# 测试 Multi-Head Attention

d_model = 16
nhead = 4
batch_size = 2
seq_len = 5

# 创建 Multi-Head Attention
mha = MultiHeadAttention(d_model, nhead)

# 模拟输入
X = torch.randn(batch_size, seq_len, d_model)

print("=" * 60)
print(f"Multi-Head Attention 测试（d_model={d_model}, nhead={nhead}")
print("=" * 60)
print(f"输入 X shape: {X.shape}")
print(f"每个头的维度 d_k = d_model / nhead = {d_model // nhead}")

# 前向传播
output, attn_weights = mha(X)

print("\n" + "=" * 60)
print("输出结果")
print("=" * 60)
print(f"输出 shape: {output.shape}")
print(f"注意力权重 shape: {attn_weights.shape}")
print(f"  → (batch_size, nhead, seq_len, seq_len)")

# 打印第 1 个样本的各头注意力权重
print("\n各头注意力权重（第 1 个样本）:")
for head in range(nhead):
    print(f"\n头 {head+1}:")
    print(torch.round(attn_weights[0, head], decimals=3))

---

# PART 5: Encoder Block（完整编码器块）

## Encoder Block 的结构

一个完整的 Encoder Block 包含：

```
输入 X
  ↓
Multi-Head Self-Attention
  ↓
残差连接（X + Attention(X)）
  ↓
LayerNorm
  ↓
Feed-Forward Network（FFN）
  ↓
残差连接（上一步输出 + FFN(上一步输出)）
  ↓
LayerNorm
  ↓
输出
```

## 残差连接和 LayerNorm

- **残差连接**：解决梯度消失问题，让信息可以直接传递
- **LayerNorm**：稳定训练，加速收敛

## FFN（前馈网络）

FFN 对每个 token 独立进行非线性变换：

FFN(x) = max(0, x @ W1 + b1) @ W2 + b2

其中 W1: (d_model, d_ff), W2: (d_ff, d_model)
d_ff 通常是 d_model 的 4 倍（Transformer 论文中 d_ff = 2048，d_model = 512）

In [ ]:
# Encoder Block 实现

class EncoderBlock(nn.Module):
    """
    完整的 Transformer Encoder Block

    Args:
        d_model: 输入维度
        nhead: 注意力头数
        dim_feedforward: FFN 隐藏层维度（默认 d_model * 4）
        dropout: dropout 概率
    """

    def __init__(self, d_model: int, nhead: int,
                 dim_feedforward: int = None, dropout: float = 0.1):
        super().__init__()

        if dim_feedforward is None:
            dim_feedforward = d_model * 4

        # 1. Multi-Head Self-Attention
        self.self_attn = MultiHeadAttention(d_model, nhead)

        # 2. LayerNorm（残差连接后）
        self.norm1 = nn.LayerNorm(d_model)

        # 3. FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, d_model)
        )

        # 4. LayerNorm（FFN 后）
        self.norm2 = nn.LayerNorm(d_model)

        # Dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """
        Args:
            X: 输入张量，shape = (batch_size, seq_len, d_model)

        Returns:
            输出张量，shape = (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, d_model = X.shape

        # ========== 子模块 1: Multi-Head Self-Attention ==========
        print(f"\n--- Encoder Block 前向传播 ---")
        print(f"输入 X shape: {X.shape}")

        attn_output, _ = self.self_attn(X)

        # 残差连接 + LayerNorm
        # X + dropout(attn_output)
        X = self.norm1(X + self.dropout(attn_output))
        print(f"\n残差连接 + LayerNorm1 后 shape: {X.shape}")

        # ========== 子模块 2: FFN ==========
        ffn_output = self.ffn(X)
        print(f"FFN 输出 shape: {ffn_output.shape}")

        # 残差连接 + LayerNorm
        X = self.norm2(X + self.dropout(ffn_output))
        print(f"残差连接 + LayerNorm2 后 shape: {X.shape}")

        return X

In [ ]:
# 测试 Encoder Block

d_model = 16
nhead = 4
dim_feedforward = 64
batch_size = 2
seq_len = 5

# 创建 Encoder Block
encoder_block = EncoderBlock(d_model, nhead, dim_feedforward, dropout=0.0)

# 模拟输入
X = torch.randn(batch_size, seq_len, d_model)

print("=" * 60)
print(f"Encoder Block 测试")
print("=" * 60)

# 前向传播
output = encoder_block(X)

print("\n" + "=" * 60)
print("输出结果")
print("=" * 60)
print(f"输出 shape: {output.shape}")
print(f"输入输出 shape 一致: {output.shape == X.shape}")

# 测试堆叠多个 Encoder Block
print("\n" + "=" * 60)
print("堆叠 2 个 Encoder Block")
print("=" * 60)

encoder_blocks = nn.ModuleList([
    EncoderBlock(d_model, nhead, dim_feedforward, dropout=0.0),
    EncoderBlock(d_model, nhead, dim_feedforward, dropout=0.0)
])

X = torch.randn(batch_size, seq_len, d_model)
for i, block in enumerate(encoder_blocks):
    print(f"\n--- 第 {i+1} 个 Encoder Block ---")
    X = block(X)

print(f"\n最终输出 shape: {X.shape}")

---

# PART 6: Decoder 核心 —— Masked Attention 和 Cross-Attention

## Decoder 与 Encoder 的区别

Decoder 比 Encoder 多一个组件：**Cross-Attention**。

### 1. Masked Self-Attention

Decoder 的 Self-Attention 需要加**因果掩码（Causal Mask）**，确保每个 token 只能关注**前面的 token**，不能关注后面的 token。

为什么？因为在生成任务中，我们是逐 token 生成的，在生成第 i 个 token 时，还不知道第 i+1 个 token 是什么。

### 2. Cross-Attention

Cross-Attention 让 Decoder 关注 Encoder 的输出：

- Q：来自 Decoder 的上一层输出
- K：来自 Encoder 的输出
- V：来自 Encoder 的输出

这就是 Decoder "理解" 输入文本的方式！

In [ ]:
# 因果掩码（Causal Mask）的构造

def create_causal_mask(seq_len: int) -> torch.Tensor:
    """
    创建因果掩码

    Args:
        seq_len: 序列长度

    Returns:
        mask: shape = (seq_len, seq_len)
              mask[i, j] = 1 表示 token i 可以关注 token j
              mask[i, j] = 0 表示 token i 不能关注 token j
    """
    # 创建上三角矩阵（包含对角线）
    # 对角线及以下为 1，对角线上方为 0
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

# 演示因果掩码
seq_len = 5
mask = create_causal_mask(seq_len)

print("=" * 60)
print("因果掩码演示（seq_len=5）")
print("=" * 60)
print(f"mask shape: {mask.shape}")
print(f"\nmask:\n{mask}")

print("\n掩码解读：")
print("  mask[i, j] = 1 → token i 可以关注 token j")
print("  mask[i, j] = 0 → token i 不能关注 token j")

print("\n例如：")
print("  token 0 只能关注 token 0（自己）")
print("  token 1 可以关注 token 0 和 token 1")
print("  token 2 可以关注 token 0, 1, 2")
print("  token 3 可以关注 token 0, 1, 2, 3")
print("  token 4 可以关注所有 token（0-4）")

# 可视化掩码
print("\n可视化：")
print("  行 = 被关注的 token")
print("  列 = 可关注的 token")
print()
for i in range(seq_len):
    row = "".join("█" if mask[i, j] == 1 else "░" for j in range(seq_len))
    print(f"  token {i}: {row}")

In [ ]:
# Cross-Attention 实现

class CrossAttention(nn.Module):
    """
    Cross-Attention 模块

    Decoder 通过 Cross-Attention 关注 Encoder 的输出

    Args:
        d_model: 输入维度
        nhead: 头数
    """

    def __init__(self, d_model: int, nhead: int):
        super().__init__()

        assert d_model % nhead == 0
        self.nhead = nhead
        self.d_k = d_model // nhead

        # Q 来自 Decoder，K/V 来自 Encoder
        self.W_Q = nn.Linear(d_model, d_model)  # Decoder → Q
        self.W_K = nn.Linear(d_model, d_model)  # Encoder → K
        self.W_V = nn.Linear(d_model, d_model)  # Encoder → V
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, decoder_output: torch.Tensor,
                encoder_output: torch.Tensor) -> torch.Tensor:
        """
        Args:
            decoder_output: Decoder 上一层的输出，shape = (B, T_dec, d_model)
            encoder_output: Encoder 的输出，shape = (B, T_enc, d_model)

        Returns:
            输出张量，shape = (B, T_dec, d_model)
        """
        batch_size, T_dec, _ = decoder_output.shape
        _, T_enc, _ = encoder_output.shape

        print(f"\n--- Cross-Attention ---")
        print(f"Decoder 输入 shape: {decoder_output.shape}")
        print(f"Encoder 输入 shape: {encoder_output.shape}")

        # Step 1: 计算 Q, K, V
        # Q 来自 Decoder
        Q = self.W_Q(decoder_output)  # (B, T_dec, d_model)
        
        # K, V 来自 Encoder
        K = self.W_K(encoder_output)  # (B, T_enc, d_model)
        V = self.W_V(encoder_output)  # (B, T_enc, d_model)

        print(f"Q (from Decoder) shape: {Q.shape}")
        print(f"K (from Encoder)  shape: {K.shape}")
        print(f"V (from Encoder)  shape: {V.shape}")

        # Step 2: 重塑为多头格式
        Q = Q.view(batch_size, T_dec, self.nhead, self.d_k).transpose(1, 2)
        K = K.view(batch_size, T_enc, self.nhead, self.d_k).transpose(1, 2)
        V = V.view(batch_size, T_enc, self.nhead, self.d_k).transpose(1, 2)

        # Step 3: 计算注意力分数
        # (B, nhead, T_dec, d_k) @ (B, nhead, d_k, T_enc)
        # → (B, nhead, T_dec, T_enc)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        print(f"scores shape: {scores.shape}")
        print(f"  → (batch, nhead, T_dec, T_enc)")

        # Step 4: softmax
        attn_weights = F.softmax(scores, dim=-1)
        print(f"attn_weights shape: {attn_weights.shape}")

        # Step 5: 加权求和
        # (B, nhead, T_dec, T_enc) @ (B, nhead, T_enc, d_k)
        # → (B, nhead, T_dec, d_k)
        output = torch.matmul(attn_weights, V)

        # Step 6: 重塑回原始格式
        output = output.transpose(1, 2).contiguous().view(batch_size, T_dec, -1)

        # Step 7: 输出投影
        output = self.W_O(output)
        print(f"最终输出 shape: {output.shape}")

        return output, attn_weights

In [ ]:
# 测试 Cross-Attention

d_model = 16
nhead = 4
batch_size = 2
T_enc = 5  # Encoder 序列长度
T_dec = 3  # Decoder 序列长度

# 创建 Cross-Attention
cross_attn = CrossAttention(d_model, nhead)

# 模拟输入
decoder_output = torch.randn(batch_size, T_dec, d_model)  # Decoder 输出
encoder_output = torch.randn(batch_size, T_enc, d_model)  # Encoder 输出

print("=" * 60)
print("Cross-Attention 测试")
print("=" * 60)

# 前向传播
output, attn_weights = cross_attn(decoder_output, encoder_output)

print("\n" + "=" * 60)
print("输出结果")
print("=" * 60)
print(f"输出 shape: {output.shape}")

# 解读注意力权重
print("\n注意力权重解读：")
print(f"attn_weights[0, 0] 表示 Decoder 的第 0 个 token 对 Encoder 各 token 的关注度")
print(f"\nDecoder token 0 对 Encoder 的注意力:\n{torch.round(attn_weights[0, 0], decimals=3)}")
print(f"\nDecoder token 1 对 Encoder 的注意力:\n{torch.round(attn_weights[0, 1], decimals=3)}")
print(f"\nDecoder token 2 对 Encoder 的注意力:\n{torch.round(attn_weights[0, 2], decimals=3)}")

---

# PART 7: Decoder Block（完整解码器块）

## Decoder Block 的结构

一个完整的 Decoder Block 包含：

```
输入 X
  ↓
Masked Multi-Head Self-Attention（因果掩码）
  ↓
残差连接 + LayerNorm
  ↓
Multi-Head Cross-Attention（Q来自Decoder，K/V来自Encoder）
  ↓
残差连接 + LayerNorm
  ↓
Feed-Forward Network
  ↓
残差连接 + LayerNorm
  ↓
输出
```

与 Encoder Block 相比，Decoder Block 多了一个 Cross-Attention 层，并且第一层 Self-Attention 需要加因果掩码。

In [ ]:
# Decoder Block 实现

class DecoderBlock(nn.Module):
    """
    完整的 Transformer Decoder Block

    Args:
        d_model: 输入维度
        nhead: 注意力头数
        dim_feedforward: FFN 隐藏层维度
        dropout: dropout 概率
    """

    def __init__(self, d_model: int, nhead: int,
                 dim_feedforward: int = None, dropout: float = 0.1):
        super().__init__()

        if dim_feedforward is None:
            dim_feedforward = d_model * 4

        # 1. Masked Multi-Head Self-Attention
        self.self_attn = MultiHeadAttention(d_model, nhead)

        # 2. LayerNorm
        self.norm1 = nn.LayerNorm(d_model)

        # 3. Cross-Attention
        self.cross_attn = CrossAttention(d_model, nhead)

        # 4. LayerNorm
        self.norm2 = nn.LayerNorm(d_model)

        # 5. FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Linear(dim_feedforward, d_model)
        )

        # 6. LayerNorm
        self.norm3 = nn.LayerNorm(d_model)

        # Dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, X: torch.Tensor, encoder_output: torch.Tensor,
                mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            X: Decoder 输入，shape = (batch_size, T_dec, d_model)
            encoder_output: Encoder 输出，shape = (batch_size, T_enc, d_model)
            mask: 因果掩码，shape = (T_dec, T_dec)

        Returns:
            输出张量，shape = (batch_size, T_dec, d_model)
        """
        print(f"\n--- Decoder Block 前向传播 ---")
        print(f"Decoder 输入 X shape: {X.shape}")
        print(f"Encoder 输出 shape: {encoder_output.shape}")

        # ========== 子模块 1: Masked Self-Attention ==========
        attn_output, _ = self.self_attn(X, mask=mask)
        X = self.norm1(X + self.dropout(attn_output))
        print(f"Masked Self-Attention 后 shape: {X.shape}")

        # ========== 子模块 2: Cross-Attention ==========
        cross_output, _ = self.cross_attn(X, encoder_output)
        X = self.norm2(X + self.dropout(cross_output))
        print(f"Cross-Attention 后 shape: {X.shape}")

        # ========== 子模块 3: FFN ==========
        ffn_output = self.ffn(X)
        X = self.norm3(X + self.dropout(ffn_output))
        print(f"FFN 后 shape: {X.shape}")

        return X

In [ ]:
# 测试 Decoder Block

d_model = 16
nhead = 4
dim_feedforward = 64
batch_size = 2
T_enc = 5  # Encoder 序列长度
T_dec = 3  # Decoder 序列长度

# 创建 Decoder Block
decoder_block = DecoderBlock(d_model, nhead, dim_feedforward, dropout=0.0)

# 模拟输入
decoder_input = torch.randn(batch_size, T_dec, d_model)
encoder_output = torch.randn(batch_size, T_enc, d_model)

# 创建因果掩码
causal_mask = create_causal_mask(T_dec)

print("=" * 60)
print("Decoder Block 测试")
print("=" * 60)

# 前向传播
output = decoder_block(decoder_input, encoder_output, mask=causal_mask)

print("\n" + "=" * 60)
print("输出结果")
print("=" * 60)
print(f"输出 shape: {output.shape}")
print(f"输入输出 shape 一致: {output.shape == decoder_input.shape}")

---

# PART 8: 完整 Transformer（Encoder + Decoder 端到端演示）

现在把所有组件组装起来，形成完整的 Transformer 模型。

完整流程：

1. **输入处理**：分词 → 词嵌入 → 位置编码
2. **Encoder**：N 个 Encoder Block 堆叠
3. **Decoder**：N 个 Decoder Block 堆叠
4. **输出层**：线性层 + softmax → 预测下一个 token

In [ ]:
# 定义位置编码（复用之前的实现）

class PositionalEncoding(nn.Module):
    """标准正余弦位置编码"""

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
# 完整 Transformer 模型

class Transformer(nn.Module):
    """
    完整的 Transformer 模型

    Args:
        src_vocab_size: 源语言词汇表大小
        tgt_vocab_size: 目标语言词汇表大小
        d_model: 模型维度
        nhead: 注意力头数
        num_encoder_layers: Encoder 层数
        num_decoder_layers: Decoder 层数
        dim_feedforward: FFN 隐藏层维度
        max_len: 最大序列长度
        dropout: dropout 概率
    """

    def __init__(self, src_vocab_size: int, tgt_vocab_size: int,
                 d_model: int = 512, nhead: int = 8,
                 num_encoder_layers: int = 6, num_decoder_layers: int = 6,
                 dim_feedforward: int = 2048, max_len: int = 5000,
                 dropout: float = 0.1):
        super().__init__()

        # ========== Encoder 部分 ==========
        # 源语言词嵌入
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)

        # 位置编码（Encoder 和 Decoder 共用）
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)

        # Encoder Blocks
        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_encoder_layers)
        ])

        # ========== Decoder 部分 ==========
        # 目标语言词嵌入
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        # Decoder Blocks
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_decoder_layers)
        ])

        # ========== 输出层 ==========
        self.output_layer = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src_tokens: torch.Tensor,
                tgt_tokens: torch.Tensor) -> torch.Tensor:
        """
        Args:
            src_tokens: 源语言 token IDs，shape = (batch_size, src_seq_len)
            tgt_tokens: 目标语言 token IDs，shape = (batch_size, tgt_seq_len)

        Returns:
            logits: shape = (batch_size, tgt_seq_len, tgt_vocab_size)
        """
        batch_size, src_seq_len = src_tokens.shape
        _, tgt_seq_len = tgt_tokens.shape

        print("=" * 60)
        print("完整 Transformer 前向传播")
        print("=" * 60)

        # ========== Encoder 前向传播 ==========
        print(f"\n【Encoder】")

        # 词嵌入 + 位置编码
        src_embedded = self.src_embedding(src_tokens) * math.sqrt(self.src_embedding.embedding_dim)
        src_embedded = self.pos_enc(src_embedded)
        print(f"源语言嵌入 shape: {src_embedded.shape}")

        # 经过所有 Encoder Block
        encoder_output = src_embedded
        for i, block in enumerate(self.encoder_blocks):
            print(f"\n--- Encoder Block {i+1} ---")
            encoder_output = block(encoder_output)

        print(f"\nEncoder 最终输出 shape: {encoder_output.shape}")

        # ========== Decoder 前向传播 ==========
        print(f"\n【Decoder】")

        # 词嵌入 + 位置编码
        tgt_embedded = self.tgt_embedding(tgt_tokens) * math.sqrt(self.tgt_embedding.embedding_dim)
        tgt_embedded = self.pos_enc(tgt_embedded)
        print(f"目标语言嵌入 shape: {tgt_embedded.shape}")

        # 创建因果掩码
        causal_mask = create_causal_mask(tgt_seq_len)

        # 经过所有 Decoder Block
        decoder_output = tgt_embedded
        for i, block in enumerate(self.decoder_blocks):
            print(f"\n--- Decoder Block {i+1} ---")
            decoder_output = block(decoder_output, encoder_output, mask=causal_mask)

        print(f"\nDecoder 最终输出 shape: {decoder_output.shape}")

        # ========== 输出层 ==========
        logits = self.output_layer(decoder_output)
        print(f"\n【输出层】")
        print(f"logits shape: {logits.shape}")

        return logits

In [ ]:
# 测试完整 Transformer

# 配置参数
src_vocab_size = 1000
tgt_vocab_size = 1000
d_model = 32
nhead = 4
num_encoder_layers = 2
num_decoder_layers = 2
dim_feedforward = 128

# 创建模型
torch.manual_seed(42)
model = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_encoder_layers,
    num_decoder_layers=num_decoder_layers,
    dim_feedforward=dim_feedforward,
    dropout=0.0
)

# 模拟输入
batch_size = 2
src_seq_len = 5
tgt_seq_len = 3

src_tokens = torch.randint(0, src_vocab_size, (batch_size, src_seq_len))
tgt_tokens = torch.randint(0, tgt_vocab_size, (batch_size, tgt_seq_len))

# 前向传播
logits = model(src_tokens, tgt_tokens)

print("\n" + "=" * 60)
print("完整 Transformer 测试结果")
print("=" * 60)
print(f"输入 src_tokens shape: {src_tokens.shape}")
print(f"输入 tgt_tokens shape: {tgt_tokens.shape}")
print(f"输出 logits    shape: {logits.shape}")
print(f"\n模型参数量: {sum(p.numel() for p in model.parameters()):,}")

---

# 总结：Encoder-Decoder 的设计哲学

## 核心架构

Transformer 采用 Encoder-Decoder 架构，适合**序列到序列**（Seq2Seq）任务：

- **Encoder**：理解输入序列，提取语义信息
- **Decoder**：基于 Encoder 的输出，生成目标序列

## Encoder 的关键组件

### 1. Multi-Head Self-Attention
- 让每个 token 关注序列中所有其他 token
- 多头并行，学习多种类型的关系

### 2. 残差连接 + LayerNorm
- 解决梯度消失问题
- 稳定训练，加速收敛

### 3. FFN
- 对每个 token 独立进行非线性变换
- 增加模型的表达能力

## Decoder 的关键组件

### 1. Masked Self-Attention
- 因果掩码：确保每个 token 只能关注前面的 token
- 保证生成过程的正确性

### 2. Cross-Attention
- Q 来自 Decoder，K/V 来自 Encoder
- Decoder 通过 Cross-Attention "理解" Encoder 的输出
- 这是 Encoder 和 Decoder 之间唯一的信息传递通道

## 数据流总结

```
源语言文本 → 分词 → 词嵌入 → 位置编码
                                      ↓
                              Encoder × N
                                      ↓
                              ┌────────┴────────┐
                              │  Cross-Attention │
                              └────────┬────────┘
                                      ↓
目标语言文本 → 分词 → 词嵌入 → 位置编码 → Decoder × N
                                                          ↓
                                                   Linear → softmax
                                                          ↓
                                                   预测下一个 token
```

## 与之前 Notebook 的关联

这个 Notebook 是 Transformer 学习系列的第三部分：

1. **tokenization_and_embedding.ipynb**：分词 + 词嵌入
2. **position_embedding.ipynb**：位置编码
3. **encoder_decoder.ipynb**：Encoder + Decoder

三个 Notebook 串联起来，构成了完整的 Transformer 输入处理和核心计算流程！

In [ ]:
# 你可以在这里实验不同的参数
# 尝试修改 nhead、num_layers 等，观察效果

# 示例：自定义实验
custom_nhead = 2
custom_num_layers = 1

custom_model = Transformer(
    src_vocab_size=1000,
    tgt_vocab_size=1000,
    d_model=16,
    nhead=custom_nhead,
    num_encoder_layers=custom_num_layers,
    num_decoder_layers=custom_num_layers,
    dropout=0.0
)

custom_src = torch.randint(0, 1000, (1, 4))
custom_tgt = torch.randint(0, 1000, (1, 3))

custom_logits = custom_model(custom_src, custom_tgt)

print(f"\n自定义实验（nhead={custom_nhead}, num_layers={custom_num_layers}）：")
print(f"  输出 shape: {custom_logits.shape}")
print(f"  参数量: {sum(p.numel() for p in custom_model.parameters()):,}")